#### **DPR**

In [1]:
%load_ext autoreload
%autoreload 2

# from src.retrievers.models.encoder import TransformerEncoder
from src.retrievers.dpr import DPR

from hydra import initialize, compose

with initialize(config_path="config", version_base=None):
    cfg = compose(config_name="config")

dpr = DPR(cfg)

/Users/rasmuslinnemann/Desktop/Courses/Semester7/02456 Deep Learning/Project/02456-retrieval-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1123 12:24:24.002000 38451 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5817.67it/s]
[transformers] BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.

Loading the dataset

In [2]:
from src.data_utils.corpus import Corpus

corpus = Corpus(dataset_name=cfg.dataset.name, base_dir=f'{cfg.base_path}/retrieval_data')

In [3]:
corpus.ds

DatasetDict({
    train: Dataset({
        features: ['question', 'text', 'query_id', 'document_id'],
        num_rows: 5500
    })
    validation: Dataset({
        features: ['question', 'text', 'query_id', 'document_id'],
        num_rows: 500
    })
    test: Dataset({
        features: ['question', 'text', 'query_id', 'document_id'],
        num_rows: 648
    })
})

Loading the queries for all 3 splits, as in `main.py`

In [4]:
from src.data_utils.query import Query

### Prepare queries
train_queries = [Query(corpus.train['question'][idx], 
                        id=corpus.train['query_id'][idx], 
                        relevant_document_ids=corpus.relevant_ids[corpus.train['query_id'][idx]]
                        ) 
                    for idx in range(corpus.train.num_rows)
                ]

val_queries =   [Query(corpus.val['question'][idx], 
                        id=corpus.val['query_id'][idx], 
                        relevant_document_ids=corpus.relevant_ids[corpus.val['query_id'][idx]]
                        ) 
                    for idx in range(corpus.val.num_rows)
                ]

test_queries =  [Query(corpus.test['question'][idx], 
                        id=corpus.test['query_id'][idx], 
                        relevant_document_ids=corpus.relevant_ids[corpus.test['query_id'][idx]]
                        ) 
                    for idx in range(corpus.test.num_rows)
                ]

From each query, we can get the ground-truth documents using `.relevant_document_ids`.

In [7]:
print(train_queries[5].text)
print(train_queries[5].relevant_document_ids)

Hobby vs. Business
['doc7', 'doc8']


Now what we are missing is to vectorize both the query and all the documents in the corpus, so that we can compare the query vector to all the document vectors and retrieve the most similar documents.

* Since we are dealing with a small dataset, we don't need a full vector database, but we can use a standard numpy array.

In [ ]:
# query    → query BERT → pool → q  (768)
# document → doc BERT   → pool → d  (768)
# score = q · d          ← dot product; high means relevant


We begin by embedding the query

* First step is to tokenize the sentence (the string is converted to numbers (e.g. tokens))

In [5]:
q_tokens = dpr.query_encoder.tokenizer(train_queries[5].text, return_tensors="pt")
print(q_tokens["input_ids"])
print(q_tokens["input_ids"].shape)

tensor([[  101, 17792,  5443,  1012,  2449,   102]])
torch.Size([1, 6])


* Second step is to run the tokenized tensor through BERT which computes attention for each token against every other token and creates a vector embedding for each token

In [6]:
q_embedding = dpr.query_encoder.model(**q_tokens)
print(q_embedding.last_hidden_state.shape)

torch.Size([1, 6, 768])


* Since we only want one vector representing the query, we perform pooling.

In [7]:
q_pooled = q_embedding.pooler_output
print(q_pooled.shape)

torch.Size([1, 768])


That is how we encode a query manually, but we have a function for it `encode_queries` which takes multiple queries as a batch.

If we want to embed the documents, we do more or less the exact same thing.

* Just for the sake of the example, we can embed the ground truth documents of q5, so we can expect a high similarity to the query

In [8]:
print(train_queries[5].id)
print(train_queries[5].relevant_document_ids)

q5
['doc7', 'doc8']


In [9]:
print(corpus.train[5]["text"][0]) #doc7
print(corpus.train[5]["text"][1]) #doc8

Miscellaneous income -- same category used for hobbies.
"You can list it as other income reported on line 21 of form 1040.  In TurboTax, enter at: - Federal Taxes tab (Personal in  Home & Business) - Wages & Income -“I’ll choose what I work on” Button Scroll down to: -Less Common Income -Misc Income, 1099-A, 1099-C. -The next screen will give you several choices. Choose ""Other reportable Income"". You will reach a screen where you can type a description of the income and the amount. Type in the amount of income and categorize as Tutoring."


Embedding doc7. This can also be done using `encode_documents`, if we want to embed multiple documents at once.

In [10]:
d_tokens = dpr.doc_encoder.tokenizer(corpus.train[5]["text"][0], return_tensors="pt")
print(d_tokens["input_ids"].shape)

d_embedding = dpr.doc_encoder.model(**d_tokens)
print(d_embedding.last_hidden_state.shape)

d_pooled = d_embedding.pooler_output
print(d_pooled.shape)

torch.Size([1, 13])
torch.Size([1, 13, 768])
torch.Size([1, 768])


Now that we have both a query and a document, we can compute the similarity

In [11]:
q_pooled @ d_pooled.T

tensor([[301.0616]], grad_fn=<MmBackward0>)

This similarity is what is computed in `query_document` too.

Now if we try to use our functions instead, on a batch of queries and a batch of documents, we could do it like this

In [12]:
query_texts = []
for idx in range(len(train_queries)):
    query_texts.append(train_queries[idx].text)
q_b10 = query_texts[0:10] # batch of 10 queries
q_b10

['What is considered a business expense on a business trip?',
 'Business Expense - Car Insurance Deductible For Accident That Occurred During a Business Trip',
 'Starting a new online business',
 '“Business day” and “due date” for bills',
 'New business owner - How do taxes work for the business vs individual?',
 'Hobby vs. Business',
 'Personal checks instead of business ones',
 'Does U.S. tax code call for small business owners to count business purchases as personal income?',
 'How can I register a UK business without providing a business address?',
 "What are 'business fundamentals'?"]

In [14]:
doc_texts = []
for idx in range(len(corpus.train)):
    doc_texts.append(corpus.train[idx]['text'][0])
d_b50 = query_texts[0:50] # batch of 50 documents
print(d_b50)

['What is considered a business expense on a business trip?', 'Business Expense - Car Insurance Deductible For Accident That Occurred During a Business Trip', 'Starting a new online business', '“Business day” and “due date” for bills', 'New business owner - How do taxes work for the business vs individual?', 'Hobby vs. Business', 'Personal checks instead of business ones', 'Does U.S. tax code call for small business owners to count business purchases as personal income?', 'How can I register a UK business without providing a business address?', "What are 'business fundamentals'?", 'Business Investment Loss from prior year', 'How can I estimate business taxes / filing fees for a business that has $0 income?', 'Would the purchase of a car for a business through the use of a business loan be considered a business expense?', 'Deducting last years (undocumented) side business loss', '30% share in business', 'Claiming business expense from personal credit card', 'Using business check to pay 

In [30]:
score_matrix = dpr.query_document(q_b10, d_b50) # B_q x B_d
print(score_matrix.shape)

torch.Size([10, 50])


In [ ]:
print(score_matrix[5])             # all document scores for q5
print(score_matrix[5].shape)
print(score_matrix[5].argmax())
print(score_matrix[5][28])         # the best match for q5 is doc28 (within this batch of documents)

tensor([320.0868, 306.3438, 306.8389, 120.1102, 277.5035, 332.2303, 255.5867,
        314.7393, 246.8532, 336.9511, 300.1353, 237.7751, 266.0819, 302.0816,
        292.9359, 311.8613, 271.6142, 260.4630, 243.0034, 257.7663, 169.6130,
        330.2973, 288.5034, 324.1734, 271.1833, 217.3333, 331.4300, 313.3962,
        363.6820, 306.4267, 280.7481, 306.2673, 303.4774, 277.2815, 317.9873,
        348.6158, 209.2316, 302.7498, 307.5723, 205.3611, 210.6670, 288.6571,
        303.0127, 273.8993, 305.7236, 267.7936, 260.3890, 262.1641, 281.4674,
        290.9100], grad_fn=<SelectBackward0>)
torch.Size([50])
tensor(28)
tensor(363.6820, grad_fn=<SelectBackward0>)


Now we are ready to start training.